In [ ]:
import shutil, os
import pandas as pd
import time
import re
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import Select

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# Clear the cached drivers
wdm_cache = os.path.expanduser("~/.wdm")
shutil.rmtree(wdm_cache, ignore_errors=True)
print("Cache cleared")

Cache cleared


In [ ]:
def accept_cookies(driver, timeout=8):
    try:
        WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable(
                (By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
            )
        ).click()
        print("Cookies accepted")
    except:
        pass


def remove_editorial_intro(paragraphs):
    if not paragraphs:
        return paragraphs

    while paragraphs:
        first = paragraphs[0].lower()

        editorial_patterns = [
            r"\bin (his|her) christmas broadcast",
            r"\bthe (king|queen)\b.{0,60}\b(broadcast|message|address|spoke|said)\b",
            r"\bthe following is the text",
            r"\bthe broadcast was",
            r"\bthis was the first",
            r"\bdelivered on\b",
            r"\baddressed the nation\b",
            r"\bchristmas broadcast\b.{0,60}\b(watched|aired|televised|recorded)\b",
            r"^(his|her) majesty",
            r"\brecorded (his|her) christmas\b",
            r"\bwere at the start of\b",
            r"\broyal yacht\b",
            r"\bgovernment house\b",
        ]

        if any(re.search(p, first) for p in editorial_patterns):
            paragraphs = paragraphs[1:]
        else:
            break

    return paragraphs


def extract_speech_body(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    containers = ["div.chapters", "div.field--name-body", "article", "main"]

    for selector in containers:
        container = soup.select_one(selector)
        if not container:
            continue

        for junk in container.select("figure, script, style, nav, header, footer"):
            junk.decompose()

        # Remove italic intro descriptions
        for tag in container.select("p em, p i"):
            parent = tag.find_parent("p")
            if parent and parent == container.find("p"):
                parent.decompose()
                break

        paragraphs = [
            p.get_text(strip=True)
            for p in container.find_all("p")
            if p.get_text(strip=True)
        ]

        paragraphs = remove_editorial_intro(paragraphs)

        if paragraphs:
            return "\n\n".join(paragraphs)

    return None


def extract_title_and_date(driver):
    soup = BeautifulSoup(driver.page_source, "html.parser")

    title = None
    for selector in ["h1.page-title", "h1.article-header__title", "h1", "title"]:
        el = soup.select_one(selector)
        if el:
            title = el.get_text(strip=True)
            title = re.sub(r"\s*[|\-–]\s*(The )?Royal.*$", "", title, flags=re.IGNORECASE).strip()
            if title:
                break

    date_obj = None

    if title:
        xmas_match = re.search(r"christmas broadcast\s+(\d{4})", title, re.IGNORECASE)
        if xmas_match:
            date_obj = datetime(int(xmas_match.group(1)), 12, 25)

    if not date_obj and title:
        m = re.search(r"(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", title)
        if m:
            for fmt in ["%d %B %Y", "%d %b %Y"]:
                try:
                    date_obj = datetime.strptime(
                        f"{m.group(1)} {m.group(2).capitalize()} {m.group(3)}", fmt
                    )
                    break
                except ValueError:
                    pass
        if not date_obj:
            m = re.search(r"([A-Za-z]+)\s+(\d{1,2}),?\s+(\d{4})", title)
            if m:
                for fmt in ["%B %d %Y", "%b %d %Y"]:
                    try:
                        date_obj = datetime.strptime(
                            f"{m.group(1).capitalize()} {m.group(2)} {m.group(3)}", fmt
                        )
                        break
                    except ValueError:
                        pass

    if not date_obj:
        date_str = None

        time_el = soup.find("time")
        if time_el:
            dt_attr = time_el.get("datetime", "")
            date_str = dt_attr if dt_attr else time_el.get_text(strip=True)

        if not date_str:
            for meta_name in ["article:published_time", "date", "DC.date"]:
                meta = soup.find("meta", {"property": meta_name}) or soup.find("meta", {"name": meta_name})
                if meta and meta.get("content"):
                    date_str = meta["content"]
                    break

        if not date_str:
            for selector in [".date", ".article-date", ".published-date", "span.date", "p.date"]:
                el = soup.select_one(selector)
                if el:
                    date_str = el.get_text(strip=True)
                    break

        if date_str:
            date_str_clean = date_str.strip().split("T")[0]
            for fmt in ["%Y-%m-%d", "%d %B %Y", "%B %d, %Y", "%d/%m/%Y", "%d %b %Y"]:
                try:
                    date_obj = datetime.strptime(date_str_clean, fmt)
                    break
                except ValueError:
                    continue
            if not date_obj:
                m = re.search(r"(\d{1,2})\s+(\w+)\s+(\d{4})", date_str)
                if m:
                    try:
                        date_obj = datetime.strptime(
                            f"{m.group(1)} {m.group(2)} {m.group(3)}", "%d %B %Y"
                        )
                    except ValueError:
                        pass

    return title, date_obj


def set_date_filter(driver, start_date_str, end_date_str):
    try:
        from_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-min']")
            )
        )
        from_input.clear()
        from_input.send_keys(start_date_str)
        from_input.send_keys(Keys.TAB)
        time.sleep(0.3)
    except Exception as e:
        print(f"  Warning: could not set min date: {e}")

    if end_date_str:
        try:
            to_input = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-max']")
                )
            )
            to_input.clear()
            to_input.send_keys(end_date_str)
            to_input.send_keys(Keys.TAB)
            time.sleep(0.8)
        except Exception as e:
            print(f"  Warning: could not set max date: {e}")


def select_monarch_filter(driver, monarch_value):
    try:
        select_el = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "edit-mrf"))
        )
        sel = Select(select_el)
        sel.select_by_value(monarch_value)
        selected_text = sel.first_selected_option.text
        print(f"  Selected monarch: {selected_text}")
    except Exception as e:
        print(f"  Warning: could not select monarch filter: {e}")


def apply_filters_and_search(driver):
    try:
        date_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "input[data-drupal-selector='edit-dates-min']")
            )
        )
        filter_form = date_input.find_element(By.XPATH, "./ancestor::form[1]")
        submit_btn = filter_form.find_element(
            By.CSS_SELECTOR, "input[type='submit']"
        )
        driver.execute_script("arguments[0].scrollIntoView(true);", submit_btn)
        time.sleep(1.0)
        driver.execute_script("arguments[0].click();", submit_btn)
        print("  Filter form submit button clicked")
        time.sleep(7)
    except Exception as e:
        print(f"  Warning: could not submit filter form: {e}")


def collect_speech_urls(driver):
    speech_urls = {}
    page_num = 1
    last_page_source = None

    while True:
        print(f"  Scanning listing page {page_num}...")

        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "li.pager__item, article, .view-content")
                )
            )
        except:
            pass
        time.sleep(5)  

        current_source = driver.page_source
        if current_source == last_page_source:
            print("  Page content unchanged — stopping to avoid infinite loop.")
            break
        last_page_source = current_source

        soup = BeautifulSoup(current_source, "html.parser")

        found_on_page = 0
        for a in soup.find_all("a", href=True):
            href = a["href"]
            title = a.get_text(strip=True)

            if any(skip in href for skip in [
                "/news/", "/photos/", "/video/", "/tags/",
                "/media-centre", "/about/", "/contact/",
                "javascript:", "#", "mailto:"
            ]):
                continue

            if href.startswith("/") and len(href) > 10:
                full_url = "https://www.royal.uk" + href
                if full_url not in speech_urls:
                    speech_urls[full_url] = title
                    found_on_page += 1

        print(f"    Found {found_on_page} new URLs on this page (total: {len(speech_urls)})")

        try:
            next_href = driver.find_element(
                By.CSS_SELECTOR, "li.pager__item--next a"
            ).get_attribute("href")
            print(f"  -> Next page: {next_href}")
            driver.get(next_href)
            time.sleep(3)  
            page_num += 1
        except:
            print(f"  No more pages. Total URLs collected: {len(speech_urls)}")
            break

    return speech_urls

In [ ]:
options = Options()
options.add_argument("--incognito")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)
driver.set_window_size(1400, 900)


START_URL = "https://www.royal.uk/media-centre/speeches"


MONARCHS_CONFIG = [
    {
        "monarch": "King Charles III",
        "filter_value": "3",                 
        "start_date": "08/09/2022",          # accession date
        "end_date": "",                      
    },
    {
        "monarch": "Queen Elizabeth II",
        "filter_value": "2",                 
        "start_date": "06/02/1952",          # accession date
        "end_date": "08/09/2022",           
    },
]

all_records = []

for config in MONARCHS_CONFIG:
    monarch_name = config["monarch"]
    print(f"\n{'='*60}")
    print(f"Scraping speeches for: {monarch_name}")
    print(f"{'='*60}")

    driver.get(START_URL)
    accept_cookies(driver)
    time.sleep(5)

    print(f"  Setting date range: {config['start_date']} → {config['end_date'] or 'present'}")
    set_date_filter(driver, config["start_date"], config["end_date"])

    print(f"  Selecting monarch filter for: {config['monarch']}")
    select_monarch_filter(driver, config["filter_value"])

    apply_filters_and_search(driver)

    speech_urls = collect_speech_urls(driver)
    print(f"  Collected {len(speech_urls)} candidate pages for {monarch_name}")

    visited_pages = set()

    for i, (url, link_text) in enumerate(speech_urls.items(), 1):
        print(f"  [{i}/{len(speech_urls)}] {link_text[:80]}")

        try:
            driver.get(url)

            canonical = driver.current_url.split("?")[0].rstrip("/")
            if canonical in visited_pages:
                print("    Duplicate alias skipped")
                continue
            visited_pages.add(canonical)

            try:
                WebDriverWait(driver, 8).until(
                    EC.presence_of_element_located(
                        (By.CSS_SELECTOR, "main, article, div.chapters")
                    )
                )
            except:
                pass

            title, date_obj = extract_title_and_date(driver)

            if not date_obj:
                year_match = re.search(r"\b(19|20)\d{2}\b", canonical + link_text)
                if year_match:
                    year = int(year_match.group())
                    date_obj = datetime(year, 1, 1)  # approximate — year only
                    print(f"    Warning: date not found on page, using year {year} from URL")

            if not date_obj:
                print("    No date found — skipped")
                continue

            if not title:
                title = link_text.strip() or f"Speech {date_obj.strftime('%Y-%m-%d')}"

            speech = extract_speech_body(driver)

            if not speech or len(speech.split()) < 80:
                print("    Too short or empty — skipped")
                continue

            all_records.append(
                {
                    "date": date_obj.strftime("%Y-%m-%d"),
                    "year": date_obj.year,
                    "monarch": monarch_name,
                    "title": title,
                    "speech": speech,
                    "url": canonical,
                }
            )

            print(f"    OK — {len(speech.split())} words | {title[:60]}")

        except Exception as e:
            print(f"    Failed: {e}")



print(f"\n{'='*60}")
print("Scraping King Charles III Christmas Broadcasts")
print(f"{'='*60}")

driver.get(START_URL)
time.sleep(5)

set_date_filter(driver, "08/09/2022", "")

select_monarch_filter(driver, "All")

try:
    search_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "input#edit-text, input[data-drupal-selector='edit-text']"))
    )
    search_box.clear()
    search_box.send_keys("Christmas broadcast")
    print("  Keyword entered: Christmas broadcast")
except Exception as e:
    print(f"  Warning: could not enter keyword: {e}")

apply_filters_and_search(driver)

christmas_urls = collect_speech_urls(driver)
print(f"  Collected {len(christmas_urls)} Christmas broadcast candidate URLs")

visited_christmas = set()
for i, (url, link_text) in enumerate(christmas_urls.items(), 1):
    print(f"  [{i}/{len(christmas_urls)}] {link_text[:80]}")
    try:
        driver.get(url)
        canonical = driver.current_url.split("?")[0].rstrip("/")
        if canonical in visited_christmas:
            print("    Duplicate alias skipped")
            continue
        visited_christmas.add(canonical)

        try:
            WebDriverWait(driver, 8).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "main, article, div.chapters"))
            )
        except:
            pass

        title, date_obj = extract_title_and_date(driver)

        if not date_obj:
            year_match = re.search(r"\b(19|20)\d{2}\b", canonical + link_text)
            if year_match:
                date_obj = datetime(int(year_match.group()), 12, 25)

        if not date_obj:
            print("    No date found — skipped")
            continue

        if not title:
            title = link_text.strip() or f"Christmas Broadcast {date_obj.year}"

        speech = extract_speech_body(driver)
        if not speech or len(speech.split()) < 80:
            print("    Too short or empty — skipped")
            continue

        all_records.append({
            "date": date_obj.strftime("%Y-%m-%d"),
            "year": date_obj.year,
            "monarch": "King Charles III",
            "title": title,
            "speech": speech,
            "url": canonical,
        })
        print(f"    OK — {len(speech.split())} words | {title[:60]}")

    except Exception as e:
        print(f"    Failed: {e}")

driver.quit()



Scraping speeches for: King Charles III
Cookies accepted
  Setting date range: 08/09/2022 → present
  Selecting monarch filter for: King Charles III
  Selected monarch: The King (formerly The Prince of Wales)
  Filter form submit button clicked
  Scanning listing page 1...
    Found 22 new URLs on this page (total: 22)
  -> Next page: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=1
  Scanning listing page 2...
    Found 10 new URLs on this page (total: 32)
  -> Next page: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=2
  Scanning listing page 3...
    Found 9 new URLs on this page (total: 41)
  -> Next page: https://www.royal.uk/media-centre/speeches?text=&mrf=3&dates%5Bmin%5D=08/09/2022&dates%5Bmax%5D=26/02/2026&page=3
  Scanning listing page 4...
    Found 10 new URLs on this page (total: 51)
  -> Next page: https://www.royal.uk/media-centre/speeches?te

In [13]:
df = pd.DataFrame(all_records).sort_values("date").reset_index(drop=True)

before = len(df)
df = df.drop_duplicates(subset=["title"], keep="first")
after = len(df)
if before > after:
    print(f"\nRemoved {before - after} duplicate title(s)")

df.to_csv("UK_monarch_speeches.csv", index=False)

print("\nDONE")
print(f"Saved {len(df)} speeches to all_UK_monarch_speeches.csv")
print(df.groupby("monarch")["title"].count().to_string())



Removed 2 duplicate title(s)

DONE
Saved 433 speeches to all_UK_monarch_speeches.csv
monarch
King Charles III       88
Queen Elizabeth II    345
